In [0]:
%sql
use catalog globalmart_dev;
use schema silver;

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import *



In [0]:
rules_customer = {
  "valid_customer_id": "customer_id IS NOT NULL",
  "valid_email": "COALESCE(TRIM(customer_email), '') <> ''",
  "valid_segment": "segment IN ('Home Office','Corporate','Consumer')",
  "valid_region": "region IN ('North','South','East','West','Central')"
}
 
quarantine_rules = "NOT({0})".format(" AND ".join(rules_customer.values()))
 
 
@dp.table(name="customers_quarantine")
def customers_quarantine():
  return (
    spark.readStream.table("customers_bronze_view")
    .withColumn("is_quarantined", expr(quarantine_rules))
    .filter("is_quarantined = true")   # ✅ only bad records
  )
 
 
@dp.table(
  name="customers",
  comment="cleaned table of the customers_bronze table"
)
@dp.expect_all(rules_customer)
def customers_silver():
  return (
    spark.readStream.table("customers_bronze_view")
    .withColumn("is_quarantined", expr(quarantine_rules))
    .filter("is_quarantined = false")   # ✅ only good records
    .withColumn(
      "region",
      when(col("region").isin("N", "North"), "North")
      .when(col("region").isin("S", "South"), "South")
      .when(col("region").isin("E", "East"), "East")
      .when(col("region").isin("W", "West"), "West")
      .when(col("region").isin("C", "Central"), "Central")
    )
    .withColumn(
      "segment",
      when(col("segment").isin("CONS", "Consumer", "Cosumer"), "Consumer")
      .when(col("segment").isin("CORP", "Corporate"), "Corporate")
      .when(col("segment").isin("HO", "Home Office"), "Home Office")
    )
  )